# Phytochemical & Antioxidant Data Analysis of Medicinal Plants

**Author:** Hafiza Muntaha Fatima  
**Focus:** Computational phytochemistry — reproducible analysis of the relationship between phenolic/flavonoid content and antioxidant activity in medicinal plants.

This notebook explores whether **total phenolic content (TPC)** and **total flavonoid content (TFC)** predict **antioxidant activity** (measured as DPPH IC50 — *lower means stronger*) across a panel of medicinal plants, including *Camellia sinensis* (green tea) and *Nigella sativa* (black seed), which are the two species in my BS research on anti-virulence therapy against *Streptococcus pyogenes*.

**Data:** `../data/medicinal_plants_phytochemistry.csv` — an educational dataset compiled from typical published assay ranges. See `../data/DATASET.md` for provenance. Values are illustrative for reproducible-analysis practice, **not** original laboratory measurements.

**Question:** Do phenolic-rich plants show stronger antioxidant activity, and are TPC and TFC correlated?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (8, 5)
pd.set_option('display.max_columns', None)

DATA = Path('..') / 'data' / 'medicinal_plants_phytochemistry.csv'
FIG = Path('..') / 'reports' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA)
print(f'Loaded {df.shape[0]} plants, {df.shape[1]} columns')
df.head()

## 1. Data validation and cleaning

Before any analysis, confirm the data types, check for missing values, and look for impossible values (e.g. negative concentrations).

In [ ]:
print('Missing values per column:')
print(df.isna().sum())

numeric_cols = ['TPC_mg_GAE_per_g', 'TFC_mg_QE_per_g', 'DPPH_IC50_ug_per_mL', 'FRAP_umol_Fe_per_g']
print('\nAny non-positive numeric values (should be none):')
print((df[numeric_cols] <= 0).sum())

print('\nDuplicate scientific names:', df['scientific_name'].duplicated().sum())

In [ ]:
df[numeric_cols].describe().round(2)

## 2. Distributions of the key measurements

Understanding the spread tells us whether the plants vary widely in antioxidant chemistry.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
titles = {
    'TPC_mg_GAE_per_g': 'Total Phenolic Content (mg GAE/g)',
    'TFC_mg_QE_per_g': 'Total Flavonoid Content (mg QE/g)',
    'DPPH_IC50_ug_per_mL': 'DPPH IC50 (ug/mL) — lower = stronger',
    'FRAP_umol_Fe_per_g': 'FRAP (umol Fe/g)'
}
for ax, col in zip(axes.ravel(), numeric_cols):
    ax.hist(df[col], bins=12, color='#4C7A34', edgecolor='white')
    ax.set_title(titles[col], fontsize=10)
    ax.set_ylabel('Number of plants')
fig.suptitle('Distribution of phytochemical and antioxidant measures', fontsize=13)
fig.tight_layout()
fig.savefig(FIG / 'distributions.png', bbox_inches='tight')
plt.show()

## 3. Do phenolics and flavonoids track antioxidant activity?

A biologically well-established expectation: higher phenolic/flavonoid content usually means **lower** DPPH IC50 (stronger radical scavenging). Let's test the correlations.

In [ ]:
corr = df[numeric_cols].corr(method='pearson')
print('Pearson correlation matrix:')
display(corr.round(3))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
short = ['TPC', 'TFC', 'DPPH IC50', 'FRAP']
ax.set_xticklabels(short, rotation=45, ha='right')
ax.set_yticklabels(short)
for i in range(len(short)):
    for j in range(len(short)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=9)
ax.set_title('Correlation heatmap')
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(FIG / 'correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, xcol, label in zip(axes, ['TPC_mg_GAE_per_g', 'TFC_mg_QE_per_g'],
                            ['Total Phenolic Content (mg GAE/g)', 'Total Flavonoid Content (mg QE/g)']):
    ax.scatter(df[xcol], df['DPPH_IC50_ug_per_mL'], color='#4C7A34', alpha=0.75)
    # trend line
    m, b = np.polyfit(df[xcol], df['DPPH_IC50_ug_per_mL'], 1)
    xs = np.linspace(df[xcol].min(), df[xcol].max(), 50)
    ax.plot(xs, m * xs + b, '--', color='#B8442A')
    r = df[xcol].corr(df['DPPH_IC50_ug_per_mL'])
    ax.set_xlabel(label)
    ax.set_ylabel('DPPH IC50 (ug/mL)')
    ax.set_title(f'r = {r:.2f}')
    # highlight the two thesis species
    for name in ['Camellia sinensis', 'Nigella sativa']:
        row = df[df['scientific_name'] == name]
        ax.scatter(row[xcol], row['DPPH_IC50_ug_per_mL'], color='#1f4e79', s=90, zorder=5)
        ax.annotate(name.split()[0], (row[xcol].values[0], row['DPPH_IC50_ug_per_mL'].values[0]),
                    textcoords='offset points', xytext=(6, 6), fontsize=9)
fig.suptitle('Antioxidant activity vs phenolic/flavonoid content (thesis species highlighted)', fontsize=12)
fig.tight_layout()
fig.savefig(FIG / 'antioxidant_vs_content.png', bbox_inches='tight')
plt.show()

## 4. Which plants are the strongest antioxidants?

Rank by DPPH IC50 (lower = stronger).

In [ ]:
top10 = df.sort_values('DPPH_IC50_ug_per_mL').head(10)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top10['plant_common_name'][::-1], top10['DPPH_IC50_ug_per_mL'][::-1], color='#4C7A34')
ax.set_xlabel('DPPH IC50 (ug/mL) — lower is stronger')
ax.set_title('Top 10 antioxidant medicinal plants in the dataset')
fig.tight_layout()
fig.savefig(FIG / 'top10_antioxidant.png', bbox_inches='tight')
plt.show()
top10[['plant_common_name', 'scientific_name', 'TPC_mg_GAE_per_g', 'DPPH_IC50_ug_per_mL']]

## 5. Family-level view

Are some botanical families richer in phenolics on average?

In [ ]:
fam = (df.groupby('family')
         .agg(n=('scientific_name', 'size'),
              mean_TPC=('TPC_mg_GAE_per_g', 'mean'),
              mean_DPPH=('DPPH_IC50_ug_per_mL', 'mean'))
         .sort_values('mean_TPC', ascending=False))
fam.round(1)

## 6. Focus on the two thesis species

How do *Camellia sinensis* and *Nigella sativa* compare to the panel? This connects the computational work back to my wet-lab research.

In [ ]:
thesis = df[df['scientific_name'].isin(['Camellia sinensis', 'Nigella sativa'])]
print('Panel median DPPH IC50:', round(df['DPPH_IC50_ug_per_mL'].median(), 1), 'ug/mL')
print('Panel median TPC     :', round(df['TPC_mg_GAE_per_g'].median(), 1), 'mg GAE/g')
thesis[['plant_common_name', 'TPC_mg_GAE_per_g', 'TFC_mg_QE_per_g', 'DPPH_IC50_ug_per_mL']]

## 7. Findings

*(Fill these in after running — example structure below; edit the numbers to match your run.)*

1. **TPC and TFC are strongly positively correlated** — phenolic-rich plants also tend to be flavonoid-rich, as expected since flavonoids are a phenolic subclass.
2. **Both TPC and TFC are negatively correlated with DPPH IC50** — higher phenolic/flavonoid content is associated with *stronger* antioxidant activity, and positively correlated with FRAP.
3. *Camellia sinensis* (green tea) sits among the strongest antioxidants in the panel, consistent with its high catechin content; *Nigella sativa* is moderate. This supports the rationale in my BS project for pairing a strong-antioxidant flavonoid source with thymoquinone.
4. Families such as Lamiaceae and Myrtaceae show high mean phenolic content.

**Reproducibility:** every figure regenerates from the CSV by running this notebook top to bottom. Data provenance is documented in `../data/DATASET.md`.

**Limitations:** this is a compiled educational dataset, not primary lab data; assay values differ across extraction methods and labs, so absolute numbers should not be over-interpreted.